# Experiment: Exp 009c Noisy Student Pooled OOF Analysis

Objective:
- Re-evaluate `exp_009` with a stricter pooled OOF view instead of optimistic fold means.
- Test whether simple postprocess or calibration variants rescue the branch.
- Decide whether `exp_009` needs calibration / stacking or should stay behind `exp_007` for Kaggle.

Success criteria:
- Produce pooled OOF metrics for raw `exp_009` and a compact family of postprocess variants.
- Compare fold-mean optimism against pooled OOF.
- Export a result snapshot and artifact tables under `experiments/outputs/exp_009c_noisy_student_pooled_oof_analysis/`.


## Plan

- Reconstruct the same validation folds used by `exp_009`.
- Load fold exports from `best_valid_meta.csv` and `best_valid_outputs.npz`.
- Score raw pooled OOF and several lightweight context / prior repairs.
- Run a fold-safe variant selection step across folds `0-2`.
- Add a small calibration view using fold-safe temperature scaling and reliability metrics.


In [ ]:
from __future__ import annotations

import json
import math
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from IPython.display import display

warnings.filterwarnings('ignore')

PROJECT_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / 'data' / 'birdclef-2026').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve PROJECT_ROOT with data/birdclef-2026')

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


@dataclass
class AnalysisConfig:
    fold_ids: tuple[int, ...] = (0, 1, 2)
    n_folds: int = 5
    lambda_event_weak: float = 0.10
    lambda_texture_weak: float = 0.20
    smooth_texture_weak: float = 0.15
    lambda_event_old: float = 0.40
    lambda_texture_old: float = 1.00
    smooth_texture_old: float = 0.35
    ece_bins: int = 15
    temperature_grid: tuple[float, ...] = (0.50, 0.75, 1.00, 1.25, 1.50, 2.00, 3.00)


CFG = AnalysisConfig()
DATA_DIR = PROJECT_ROOT / 'data' / 'birdclef-2026'
EXP009_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_009_noisy_student_pseudolabel'
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_009c_noisy_student_pooled_oof_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL_TO_IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
TEXTURE_TAXA = {'Amphibia', 'Insecta'}
TEXTURE_SET = set(taxonomy.loc[taxonomy['class_name'].isin(TEXTURE_TAXA), 'primary_label'].tolist())
IDX_TEXTURE = np.array([LABEL_TO_IDX[label] for label in PRIMARY_LABELS if label in TEXTURE_SET], dtype=np.int32)
IDX_ALL = np.arange(len(PRIMARY_LABELS), dtype=np.int32)
IDX_EVENT = np.array([idx for idx, label in enumerate(PRIMARY_LABELS) if label not in TEXTURE_SET], dtype=np.int32)

{
    'project_root': str(PROJECT_ROOT),
    'exp009_dir': str(EXP009_DIR),
    'output_dir': str(OUTPUT_DIR),
    **asdict(CFG),
}


In [ ]:
def parse_soundscape_labels(value) -> list[str]:
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    return sorted(set(label for item in series for label in parse_soundscape_labels(item)))


def parse_soundscape_filename(name: str) -> dict:
    stem = Path(name).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return {'site': site, 'hour_utc': hour_utc}


def encode_multi_hot(labels: list[str]) -> np.ndarray:
    target = np.zeros(len(PRIMARY_LABELS), dtype=np.float32)
    for label in labels:
        idx = LABEL_TO_IDX.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


def stable_logit(p: np.ndarray, eps: float = 1e-4) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=np.float32), eps, 1.0 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def stable_sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    positive = x >= 0
    out = np.empty_like(x, dtype=np.float32)
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def score_macro_auc(y_true: np.ndarray, y_score: np.ndarray, labels: list[str]) -> dict:
    aucs = []
    skipped_no_positive = 0
    skipped_no_negative = 0
    for idx, _ in enumerate(labels):
        positives = int(y_true[:, idx].sum())
        negatives = int(len(y_true) - positives)
        if positives == 0:
            skipped_no_positive += 1
            continue
        if negatives == 0:
            skipped_no_negative += 1
            continue
        aucs.append(float(roc_auc_score(y_true[:, idx], y_score[:, idx])))
    macro_auc = float(np.mean(aucs)) if aucs else float('nan')
    return {
        'macro_auc': macro_auc,
        'scored_classes': len(aucs),
        'skipped_no_positive': skipped_no_positive,
        'skipped_no_negative': skipped_no_negative,
    }


def brier_score_multilabel(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    return float(np.mean((y_true.astype(np.float32) - y_prob.astype(np.float32)) ** 2))


def binary_log_loss_from_probs(y_true: np.ndarray, y_prob: np.ndarray, eps: float = 1e-6) -> float:
    y_prob = np.clip(y_prob.astype(np.float32), eps, 1.0 - eps)
    y_true = y_true.astype(np.float32)
    loss = -(y_true * np.log(y_prob) + (1.0 - y_true) * np.log(1.0 - y_prob))
    return float(loss.mean())


def binary_log_loss_from_logits(y_true: np.ndarray, logits: np.ndarray, eps: float = 1e-6) -> float:
    return binary_log_loss_from_probs(y_true, stable_sigmoid(logits), eps=eps)


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    probs = y_prob.reshape(-1).astype(np.float32)
    targets = y_true.reshape(-1).astype(np.float32)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    total = len(probs)
    for idx in range(n_bins):
        left = bin_edges[idx]
        right = bin_edges[idx + 1]
        if idx == n_bins - 1:
            mask = (probs >= left) & (probs <= right)
        else:
            mask = (probs >= left) & (probs < right)
        if not np.any(mask):
            continue
        conf = float(probs[mask].mean())
        acc = float(targets[mask].mean())
        ece += (mask.sum() / total) * abs(acc - conf)
    return float(ece)


def classwise_auc_table(y_true: np.ndarray, y_score: np.ndarray, labels: list[str]) -> pd.DataFrame:
    rows = []
    for idx, label in enumerate(labels):
        positives = int(y_true[:, idx].sum())
        negatives = int(len(y_true) - positives)
        if positives == 0 or negatives == 0:
            rows.append({'primary_label': label, 'auc': np.nan, 'positives': positives, 'negatives': negatives})
        else:
            rows.append({
                'primary_label': label,
                'auc': float(roc_auc_score(y_true[:, idx], y_score[:, idx])),
                'positives': positives,
                'negatives': negatives,
            })
    return pd.DataFrame(rows)


def fit_prior_tables(prior_df: pd.DataFrame, y_prior: np.ndarray) -> dict:
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    for site in site_keys:
        idx = site_to_i[site]
        mask = prior_df['site'].astype(str).values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = prior_df['hour_utc'].astype(int).values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (site, hour), group_idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(site), int(hour))] = len(sh_n_list)
        group_idx = np.array(list(group_idx))
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, y_prior.shape[1]), dtype=np.float32)

    return {
        'global_p': global_p,
        'site_to_i': site_to_i,
        'site_n': site_n,
        'site_p': site_p,
        'hour_to_i': hour_to_i,
        'hour_n': hour_n,
        'hour_p': hour_p,
        'sh_to_i': sh_to_i,
        'sh_n': sh_n,
        'sh_p': sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps: float = 1e-4):
    n_rows = len(sites)
    p = np.repeat(tables['global_p'][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables['site_to_i'].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables['sh_to_i'].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return stable_logit(p, eps=eps)


def smooth_cols_by_file(scores: np.ndarray, meta_df: pd.DataFrame, cols: np.ndarray, alpha: float) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    smoothed = scores.copy()
    for _, group_idx in meta_df.groupby('filename', sort=False).groups.items():
        group_idx = np.array(list(group_idx), dtype=np.int32)
        group_scores = smoothed[group_idx][:, cols]
        blended = group_scores.copy()
        if len(group_idx) > 1:
            blended[1:] += alpha * group_scores[:-1]
            blended[:-1] += alpha * group_scores[1:]
            denom = np.ones((len(group_idx), 1), dtype=np.float32)
            denom[1:] += alpha
            denom[:-1] += alpha
            blended /= denom
        smoothed[np.ix_(group_idx, cols)] = blended
    return smoothed


def filemax_cols_by_file(scores: np.ndarray, meta_df: pd.DataFrame, cols: np.ndarray, alpha: float) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    blended = scores.copy()
    for _, group_idx in meta_df.groupby('filename', sort=False).groups.items():
        group_idx = np.array(list(group_idx), dtype=np.int32)
        group_scores = blended[group_idx][:, cols]
        file_max = group_scores.max(axis=0, keepdims=True)
        group_scores = (1.0 - alpha) * group_scores + alpha * file_max
        blended[np.ix_(group_idx, cols)] = group_scores
    return blended


def fuse_native_logits(logits: np.ndarray, prior_logits: np.ndarray, meta_df: pd.DataFrame, lambda_event: float, lambda_texture: float, smooth_texture: float) -> np.ndarray:
    fused = logits.copy()
    if len(IDX_EVENT):
        fused[:, IDX_EVENT] += lambda_event * prior_logits[:, IDX_EVENT]
    if len(IDX_TEXTURE):
        fused[:, IDX_TEXTURE] += lambda_texture * prior_logits[:, IDX_TEXTURE]
        fused = smooth_cols_by_file(fused, meta_df, IDX_TEXTURE, alpha=smooth_texture)
    return fused


def context_repair_logits(logits: np.ndarray, meta_df: pd.DataFrame, event_smooth: float, texture_filemax: float) -> np.ndarray:
    repaired = logits.copy()
    if len(IDX_EVENT):
        repaired = smooth_cols_by_file(repaired, meta_df, IDX_EVENT, alpha=event_smooth)
    if len(IDX_TEXTURE):
        repaired = filemax_cols_by_file(repaired, meta_df, IDX_TEXTURE, alpha=texture_filemax)
    return repaired


def choose_temperature(logits: np.ndarray, y_true: np.ndarray, temperature_grid: tuple[float, ...]) -> float:
    best_t = 1.0
    best_loss = math.inf
    for temp in temperature_grid:
        loss = binary_log_loss_from_logits(y_true, logits / temp)
        if loss < best_loss:
            best_loss = loss
            best_t = float(temp)
    return float(best_t)


def collect_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    metrics = score_macro_auc(y_true, y_prob, PRIMARY_LABELS)
    metrics['brier'] = brier_score_multilabel(y_true, y_prob)
    metrics['log_loss'] = binary_log_loss_from_probs(y_true, y_prob)
    metrics['ece'] = expected_calibration_error(y_true, y_prob, n_bins=CFG.ece_bins)
    return metrics


In [ ]:
sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
full_df = sc_clean[sc_clean['filename'].isin(full_files)].reset_index(drop=True)

n_group_splits = min(CFG.n_folds, max(2, len(full_files)))
gkf = GroupKFold(n_splits=n_group_splits)
full_splits = list(gkf.split(full_df, groups=full_df['filename']))

variant_settings = {
    'raw': ('raw', {}),
    'weak_priors': ('priors', {'lambda_event': CFG.lambda_event_weak, 'lambda_texture': CFG.lambda_texture_weak, 'smooth_texture': CFG.smooth_texture_weak}),
    'old_priors': ('priors', {'lambda_event': CFG.lambda_event_old, 'lambda_texture': CFG.lambda_texture_old, 'smooth_texture': CFG.smooth_texture_old}),
    'filemax_all_020': ('filemax', {'alpha': 0.20}),
    'filemax_all_035': ('filemax', {'alpha': 0.35}),
    'smooth_all_020': ('smooth', {'alpha': 0.20}),
    'event_smooth_texture_filemax': ('context', {'event_smooth': 0.20, 'texture_filemax': 0.35}),
    'event_smooth_texture_filemax_strong': ('context', {'event_smooth': 0.35, 'texture_filemax': 0.50}),
}

fold_rows = []
variant_parts = {name: [] for name in variant_settings}
raw_logits_by_fold = {}
fold_exports = []

for fold in CFG.fold_ids:
    train_idx, valid_idx = full_splits[int(fold)]
    valid_files = set(full_df.iloc[valid_idx]['filename'].tolist())

    train_df = (
        sc_clean[~sc_clean['filename'].isin(valid_files)]
        .sort_values(['filename', 'end_sec'])
        .reset_index(drop=True)
    )
    valid_df = (
        full_df[full_df['filename'].isin(valid_files)]
        .sort_values(['filename', 'end_sec'])
        .reset_index(drop=True)
    )

    fold_dir = EXP009_DIR / f'fold_{int(fold):02d}'
    snapshot = json.loads((fold_dir / 'result_snapshot.json').read_text())
    meta_df = pd.read_csv(fold_dir / 'best_valid_meta.csv').sort_values(['filename', 'end_sec']).reset_index(drop=True)
    outputs = np.load(fold_dir / 'best_valid_outputs.npz')
    y_true = outputs['y_true'].astype(np.float32)
    raw_probs = outputs['y_pred'].astype(np.float32)
    raw_logits = stable_logit(raw_probs)

    if len(meta_df) != len(valid_df):
        raise ValueError(f'Fold {fold}: exported meta rows do not match reconstructed valid rows: {len(meta_df)} vs {len(valid_df)}')
    if meta_df['row_id'].tolist() != valid_df['row_id'].tolist():
        raise ValueError(f'Fold {fold}: row_id mismatch between exported meta and reconstructed validation fold')

    y_train = np.stack([encode_multi_hot(labels) for labels in train_df['label_list']], axis=0)
    prior_tables = fit_prior_tables(train_df[['site', 'hour_utc']], y_train)
    prior_logits = prior_logits_from_tables(
        sites=meta_df['site'].to_numpy(),
        hours=meta_df['hour_utc'].to_numpy(),
        tables=prior_tables,
    )

    raw_logits_by_fold[int(fold)] = {'meta': meta_df.copy(), 'y_true': y_true.copy(), 'raw_logits': raw_logits.copy()}

    for name, (kind, params) in variant_settings.items():
        if kind == 'raw':
            probs = raw_probs.copy()
        elif kind == 'priors':
            logits = fuse_native_logits(raw_logits, prior_logits, meta_df, **params)
            probs = stable_sigmoid(logits)
        elif kind == 'filemax':
            logits = filemax_cols_by_file(raw_logits, meta_df, IDX_ALL, alpha=float(params['alpha']))
            probs = stable_sigmoid(logits)
        elif kind == 'smooth':
            logits = smooth_cols_by_file(raw_logits, meta_df, IDX_ALL, alpha=float(params['alpha']))
            probs = stable_sigmoid(logits)
        elif kind == 'context':
            logits = context_repair_logits(raw_logits, meta_df, **params)
            probs = stable_sigmoid(logits)
        else:
            raise ValueError(f'Unknown variant kind: {kind}')

        metrics = collect_metrics(y_true, probs)
        fold_rows.append({'fold': int(fold), 'variant': name, **metrics})
        variant_parts[name].append((int(fold), meta_df.copy(), y_true.copy(), probs.copy()))

    fold_export = meta_df.copy()
    fold_export['fold'] = int(fold)
    fold_export['source_best_epoch'] = int(snapshot.get('best_epoch')) if snapshot.get('best_epoch') is not None else None
    fold_export['raw_confidence'] = raw_probs.max(axis=1)
    fold_exports.append(fold_export)

fold_results = pd.DataFrame(fold_rows).sort_values(['fold', 'macro_auc'], ascending=[True, False]).reset_index(drop=True)
fold_results.to_csv(OUTPUT_DIR / 'fold_results.csv', index=False)
print(fold_results)


In [ ]:
oof_rows = []
variant_predictions = {}
variant_targets = None
variant_meta = None

for name, parts in variant_parts.items():
    parts = sorted(parts, key=lambda item: item[0])
    meta_concat = pd.concat([item[1] for item in parts], axis=0).reset_index(drop=True)
    y_true_concat = np.concatenate([item[2] for item in parts], axis=0)
    y_pred_concat = np.concatenate([item[3] for item in parts], axis=0)
    metrics = collect_metrics(y_true_concat, y_pred_concat)
    oof_rows.append({'variant': name, **metrics})
    variant_predictions[name] = y_pred_concat
    if variant_targets is None:
        variant_targets = y_true_concat
        variant_meta = meta_concat

oof_ablation = pd.DataFrame(oof_rows).sort_values('macro_auc', ascending=False).reset_index(drop=True)
oof_ablation.to_csv(OUTPUT_DIR / 'oof_ablation_results.csv', index=False)

best_fixed_variant = str(oof_ablation.iloc[0]['variant'])
raw_probs_oof = variant_predictions['raw']
best_fixed_probs_oof = variant_predictions[best_fixed_variant]

selection_rows = []
selected_parts = []
temp_scaled_parts = []
chosen_variants = {}
chosen_temperatures = {}

for target_fold in CFG.fold_ids:
    train_folds = [int(fold) for fold in CFG.fold_ids if int(fold) != int(target_fold)]
    train_scores = []
    for name, parts in variant_parts.items():
        train_parts = [item for item in parts if int(item[0]) in train_folds]
        y_true_train = np.concatenate([item[2] for item in train_parts], axis=0)
        y_pred_train = np.concatenate([item[3] for item in train_parts], axis=0)
        metrics = collect_metrics(y_true_train, y_pred_train)
        train_scores.append({'target_fold': int(target_fold), 'variant': name, **metrics})
    train_scores_df = pd.DataFrame(train_scores).sort_values('macro_auc', ascending=False).reset_index(drop=True)
    chosen_variant = str(train_scores_df.iloc[0]['variant'])
    chosen_variants[int(target_fold)] = chosen_variant
    selection_rows.append({
        'target_fold': int(target_fold),
        'chosen_variant': chosen_variant,
        'train_macro_auc': float(train_scores_df.iloc[0]['macro_auc']),
        'train_brier': float(train_scores_df.iloc[0]['brier']),
        'train_ece': float(train_scores_df.iloc[0]['ece']),
    })

    target_part = [item for item in variant_parts[chosen_variant] if int(item[0]) == int(target_fold)][0]
    selected_parts.append(target_part)

    raw_train_logits = np.concatenate([raw_logits_by_fold[int(fold)]['raw_logits'] for fold in train_folds], axis=0)
    raw_train_true = np.concatenate([raw_logits_by_fold[int(fold)]['y_true'] for fold in train_folds], axis=0)
    chosen_temp = choose_temperature(raw_train_logits, raw_train_true, CFG.temperature_grid)
    chosen_temperatures[int(target_fold)] = chosen_temp

    raw_target = raw_logits_by_fold[int(target_fold)]
    temp_probs = stable_sigmoid(raw_target['raw_logits'] / chosen_temp)
    temp_scaled_parts.append((int(target_fold), raw_target['meta'].copy(), raw_target['y_true'].copy(), temp_probs.copy()))

selection_df = pd.DataFrame(selection_rows)
selection_df['chosen_temperature'] = selection_df['target_fold'].map(chosen_temperatures)
selection_df.to_csv(OUTPUT_DIR / 'fold_safe_variant_selection.csv', index=False)

selected_parts = sorted(selected_parts, key=lambda item: item[0])
y_true_selected = np.concatenate([item[2] for item in selected_parts], axis=0)
y_pred_selected = np.concatenate([item[3] for item in selected_parts], axis=0)
meta_selected = pd.concat([item[1] for item in selected_parts], axis=0).reset_index(drop=True)
selected_metrics = collect_metrics(y_true_selected, y_pred_selected)

temp_scaled_parts = sorted(temp_scaled_parts, key=lambda item: item[0])
y_true_temp = np.concatenate([item[2] for item in temp_scaled_parts], axis=0)
y_pred_temp = np.concatenate([item[3] for item in temp_scaled_parts], axis=0)
temp_metrics = collect_metrics(y_true_temp, y_pred_temp)

calibration_rows = [
    {'strategy': 'raw', **collect_metrics(variant_targets, raw_probs_oof)},
    {'strategy': f'best_fixed::{best_fixed_variant}', **collect_metrics(variant_targets, best_fixed_probs_oof)},
    {'strategy': 'fold_safe_selected', **selected_metrics},
    {'strategy': 'fold_safe_temperature_scaled_raw', **temp_metrics},
]
calibration_df = pd.DataFrame(calibration_rows).sort_values('macro_auc', ascending=False).reset_index(drop=True)
calibration_df.to_csv(OUTPUT_DIR / 'calibration_results.csv', index=False)

classwise_raw = classwise_auc_table(variant_targets, raw_probs_oof, PRIMARY_LABELS).rename(columns={'auc': 'raw_auc'})
classwise_best = classwise_auc_table(y_true_selected, y_pred_selected, PRIMARY_LABELS).rename(columns={'auc': 'selected_auc'})
classwise_compare = classwise_raw.merge(classwise_best[['primary_label', 'selected_auc']], on='primary_label', how='left')
classwise_compare['delta'] = classwise_compare['selected_auc'] - classwise_compare['raw_auc']
classwise_compare.to_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv', index=False)

export_name = 'fold_safe_selected'
best_export = pd.concat([
    meta_selected.reset_index(drop=True),
    pd.DataFrame({'strategy': [export_name] * len(meta_selected)}),
    pd.DataFrame(y_true_selected.astype(np.int8), columns=[f'true__{label}' for label in PRIMARY_LABELS]),
    pd.DataFrame(raw_probs_oof.astype(np.float32), columns=[f'raw__{label}' for label in PRIMARY_LABELS]),
    pd.DataFrame(y_pred_selected.astype(np.float32), columns=[f'{export_name}__{label}' for label in PRIMARY_LABELS]),
], axis=1)
try:
    best_export.to_parquet(OUTPUT_DIR / 'best_oof_predictions.parquet', index=False)
except Exception:
    best_export.to_csv(OUTPUT_DIR / 'best_oof_predictions.csv', index=False)

pd.concat(fold_exports, axis=0).to_csv(OUTPUT_DIR / 'fold_meta_overview.csv', index=False)

result_snapshot = {
    'experiment_id': 'exp_009c',
    'experiment_name': 'noisy_student_pooled_oof_analysis',
    'fold_ids': [int(fold) for fold in CFG.fold_ids],
    'raw_pooled_oof_macro_auc': float(oof_ablation.loc[oof_ablation['variant'] == 'raw', 'macro_auc'].iloc[0]),
    'raw_pooled_oof_brier': float(oof_ablation.loc[oof_ablation['variant'] == 'raw', 'brier'].iloc[0]),
    'raw_pooled_oof_ece': float(oof_ablation.loc[oof_ablation['variant'] == 'raw', 'ece'].iloc[0]),
    'raw_fold_mean_macro_auc': float(fold_results[fold_results['variant'] == 'raw']['macro_auc'].mean()),
    'optimism_gap_fold_mean_minus_pooled': float(fold_results[fold_results['variant'] == 'raw']['macro_auc'].mean() - oof_ablation.loc[oof_ablation['variant'] == 'raw', 'macro_auc'].iloc[0]),
    'best_fixed_variant': best_fixed_variant,
    'best_fixed_variant_macro_auc': float(oof_ablation.iloc[0]['macro_auc']),
    'fold_safe_selected_macro_auc': float(selected_metrics['macro_auc']),
    'fold_safe_selected_brier': float(selected_metrics['brier']),
    'fold_safe_selected_ece': float(selected_metrics['ece']),
    'temperature_scaled_macro_auc': float(temp_metrics['macro_auc']),
    'temperature_scaled_brier': float(temp_metrics['brier']),
    'temperature_scaled_ece': float(temp_metrics['ece']),
    'chosen_variants_by_fold': {str(k): v for k, v in chosen_variants.items()},
    'chosen_temperatures_by_fold': {str(k): float(v) for k, v in chosen_temperatures.items()},
    'raw_vs_exp007_gap': float(oof_ablation.loc[oof_ablation['variant'] == 'raw', 'macro_auc'].iloc[0] - 0.7109),
}
(OUTPUT_DIR / 'result_snapshot.json').write_text(json.dumps(result_snapshot, indent=2))

print(oof_ablation)
print()
print('Fold-safe selection:')
print(selection_df)
print()
print('Calibration summary:')
print(calibration_df)
print()
print('Snapshot:')
print(json.dumps(result_snapshot, indent=2))


In [ ]:
snapshot = json.loads((OUTPUT_DIR / 'result_snapshot.json').read_text())
print('Snapshot:')
print(json.dumps(snapshot, indent=2))

print('\nFold results:')
display(pd.read_csv(OUTPUT_DIR / 'fold_results.csv'))

print('\nOOF ablations:')
display(pd.read_csv(OUTPUT_DIR / 'oof_ablation_results.csv'))

print('\nFold-safe variant selection:')
display(pd.read_csv(OUTPUT_DIR / 'fold_safe_variant_selection.csv'))

print('\nCalibration results:')
display(pd.read_csv(OUTPUT_DIR / 'calibration_results.csv'))

compare_df = pd.read_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv')
print('\nTop gains vs raw:')
display(compare_df.sort_values('delta', ascending=False).head(20))
print('\nTop regressions vs raw:')
display(compare_df.sort_values('delta', ascending=True).head(20))
